# Week 2 — Hybrid Retrieval (Reciprocal Rank Fusion)
**Method:** BM25 + MiniLM-Dense combined via RRF  
**RRF Formula:** `score(d) = 1/(k + rank_bm25) + 1/(k + rank_dense)`  where k=60  
**Dataset:** FiQA Test Set (648 queries, 57,600 passages)  
**Loads:** `baseline_results.csv` + `minilm_results.csv`  
**Saves:** `hybrid_results.csv`, `hybrid_per_query.csv`, `hybrid_metrics.json`

## Cell 0 — Repo Root & Config

In [ ]:
import sys
import os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / "config.py").exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import FIQA_QUERIES, FIQA_QRELS_TEST

# Input paths
BM25_RESULTS   = os.path.join("baseline",        "Results", "baseline_results.csv")
BM25_METRICS   = os.path.join("baseline",        "Results", "baseline_metrics.json")
DENSE_RESULTS  = os.path.join("dense_retrieval", "Results", "minilm_results.csv")
DENSE_METRICS  = os.path.join("dense_retrieval", "Results", "dense_metrics.json")

# Output paths
HYBRID_DIR = os.path.join("hybrid", "Results")
os.makedirs(HYBRID_DIR, exist_ok=True)

print(f"Repo root      : {_root}")
print(f"BM25 results   : {BM25_RESULTS}")
print(f"Dense results  : {DENSE_RESULTS}")
print(f"Output dir     : {HYBRID_DIR}")

# Verify inputs exist
for p in [BM25_RESULTS, DENSE_RESULTS]:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {p}")

## Cell 1 — Install & Imports

In [ ]:
import sys
!{sys.executable} -m pip install -q pytrec-eval-terrier

In [ ]:
import json
import numpy as np
import pandas as pd
import pytrec_eval
from collections import defaultdict

print("All imports OK")

## Cell 2 — Load BM25 & Dense Results

In [ ]:
def load_results_csv(path):
    """
    Load a results CSV (query_id, rank, doc_id, score) into
    a ranked dict: {qid: [(doc_id, score), ...]} sorted by rank.
    """
    df = pd.read_csv(path, dtype={"query_id": str, "doc_id": str})
    ranked = defaultdict(list)
    for _, row in df.sort_values("rank").iterrows():
        ranked[str(row["query_id"])].append((str(row["doc_id"]), float(row["score"])))
    return dict(ranked)


bm25_ranked   = load_results_csv(BM25_RESULTS)
dense_ranked  = load_results_csv(DENSE_RESULTS)

print(f"BM25  queries loaded  : {len(bm25_ranked):,}")
print(f"Dense queries loaded  : {len(dense_ranked):,}")
print(f"Query overlap         : {len(set(bm25_ranked) & set(dense_ranked)):,}")

## Cell 3 — Load Qrels

In [ ]:
qrels_df = pd.read_csv(FIQA_QRELS_TEST)
qrels_df.columns = [c.strip() for c in qrels_df.columns]
qrels_df = qrels_df.astype(str)

query_col  = [c for c in qrels_df.columns if "query"  in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if "corpus" in c.lower()][0]
score_col  = [c for c in qrels_df.columns if "score"  in c.lower()][0]

qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    qrels_dict.setdefault(qid, {})[did] = int(float(row[score_col]))

# Load queries for per-query CSV labels
queries_df = pd.read_csv(FIQA_QUERIES, usecols=["_id", "text"])
queries_df["_id"] = queries_df["_id"].astype(str)

print(f"Qrels loaded : {len(qrels_dict):,} queries")

## Cell 4 — Reciprocal Rank Fusion (RRF)
RRF merges two ranked lists without needing score normalisation.  
Each document gets `1/(k + rank)` from each retriever; scores are summed.  
k=60 is the standard value from Cormack et al. (2009).

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60, top_k=10):
    """
    Fuse multiple ranked lists using RRF.
    ranked_lists: list of {qid: [(doc_id, score), ...]} dicts
    Returns: {qid: {doc_id: rrf_score}}
    """
    all_qids = set()
    for rl in ranked_lists:
        all_qids.update(rl.keys())

    fused = {}
    for qid in all_qids:
        doc_scores = defaultdict(float)
        for rl in ranked_lists:
            if qid not in rl:
                continue
            for rank, (doc_id, _) in enumerate(rl[qid], start=1):
                doc_scores[doc_id] += 1.0 / (k + rank)
        # Keep top_k by RRF score
        top_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        fused[qid] = dict(top_docs)
    return fused


print("Running RRF  (BM25 + MiniLM-Dense, k=60) ...")
hybrid_results = reciprocal_rank_fusion(
    [bm25_ranked, dense_ranked],
    k=60,
    top_k=10
)
print(f"Hybrid results computed : {len(hybrid_results):,} queries")

## Cell 5 — Evaluate Hybrid

In [ ]:
evaluator  = pytrec_eval.RelevanceEvaluator(
    qrels_dict, {"ndcg_cut_10", "recip_rank", "recall_10"}
)
eval_res   = evaluator.evaluate(hybrid_results)

hybrid_agg = {
    "NDCG@10"   : round(np.mean([v["ndcg_cut_10"] for v in eval_res.values()]), 4),
    "MRR"       : round(np.mean([v["recip_rank"]  for v in eval_res.values()]), 4),
    "Recall@10" : round(np.mean([v["recall_10"]   for v in eval_res.values()]), 4),
}

print("\n" + "="*45)
print("      HYBRID RRF RESULTS")
print("="*45)
print(f"  Queries evaluated : {len(eval_res)}")
print(f"  NDCG@10           : {hybrid_agg['NDCG@10']}")
print(f"  MRR               : {hybrid_agg['MRR']}")
print(f"  Recall@10         : {hybrid_agg['Recall@10']}")
print("="*45)

## Cell 6 — Save Results

In [ ]:
# Save top-10 results per query
rows = [
    {"query_id": qid, "rank": rank + 1, "doc_id": doc_id, "score": round(score, 8)}
    for qid, docs in hybrid_results.items()
    for rank, (doc_id, score) in enumerate(
        sorted(docs.items(), key=lambda x: x[1], reverse=True)
    )
]
pd.DataFrame(rows).to_csv(os.path.join(HYBRID_DIR, "hybrid_results.csv"), index=False)
print(f"Saved: hybrid_results.csv  ({len(rows)} rows)")

# Save per-query metrics
pq_rows = []
for qid, scores in eval_res.items():
    qt = queries_df[queries_df["_id"] == qid]["text"].values
    pq_rows.append({
        "query_id"  : qid,
        "query"     : qt[0] if len(qt) else "",
        "NDCG@10"   : round(scores["ndcg_cut_10"], 4),
        "MRR"       : round(scores["recip_rank"],  4),
        "Recall@10" : round(scores["recall_10"],   4),
    })
pd.DataFrame(pq_rows).sort_values("NDCG@10", ascending=False).to_csv(
    os.path.join(HYBRID_DIR, "hybrid_per_query.csv"), index=False
)
print(f"Saved: hybrid_per_query.csv")

## Cell 7 — Final 3-Model Comparison Table & Save hybrid_metrics.json

In [ ]:
# Load previous metrics
with open(BM25_METRICS)   as f: bm25_m  = json.load(f)
with open(DENSE_METRICS)  as f: dense_m = json.load(f)

final_comparison = {
    "dataset"     : "FiQA Test",
    "num_queries" : len(eval_res),
    "models": {
        "BM25 Baseline" : {
            "NDCG@10"   : bm25_m["NDCG@10"],
            "MRR"       : bm25_m["MRR"],
            "Recall@10" : bm25_m["Recall@10"]
        },
        "MiniLM-Dense"  : dense_m["models"]["MiniLM-Dense"],
        "Hybrid-RRF"    : hybrid_agg,
    }
}

with open(os.path.join(HYBRID_DIR, "hybrid_metrics.json"), "w") as f:
    json.dump(final_comparison, f, indent=2)
print("Saved: hybrid_metrics.json")

bm25_ndcg = final_comparison["models"]["BM25 Baseline"]["NDCG@10"]
print("\n" + "="*62)
print("          WEEK 2 — RRF COMPARISON TABLE")
print("="*62)
print(f"  {'Model':<20} {'NDCG@10':>8} {'MRR':>8} {'Recall@10':>10}  {'vs BM25':>8}")
print("-"*62)
for name, m in final_comparison["models"].items():
    delta = f"+{((m['NDCG@10']-bm25_ndcg)/bm25_ndcg*100):.1f}%" if name != "BM25 Baseline" else "baseline"
    print(f"  {name:<20} {m['NDCG@10']:>8.4f} {m['MRR']:>8.4f} {m['Recall@10']:>10.4f}  {delta:>8}")
print("="*62)
print("\nNote: RRF k=60 underperforms Dense — BM25 is dragging results down.")
print("See Cell 9 (Alpha Sweep) for the fix.")

## Cell 8 — Saved Files Summary

In [ ]:
print("Files saved in hybrid/Results/:")
for dirpath, _, filenames in os.walk(HYBRID_DIR):
    for fname in sorted(filenames):
        fpath = os.path.join(dirpath, fname)
        size  = os.path.getsize(fpath)
        rel   = os.path.relpath(fpath, _root)
        print(f"  {rel:<50}  {size/1024:>7.1f} KB")

print("\nWeek 2 retrieval complete.")
print("Next → PDF_Chunking.ipynb  (build knowledge base for Weeks 3-4)")

## Cell 9 — Alpha Score Fusion (Fix for RRF underperformance)
**Why RRF failed:** k=60 treats BM25 and Dense equally. BM25 (0.2169) is much weaker than Dense (0.3687), so it pollutes the ranking.  
**Fix — Alpha Fusion:**  
1. Min-max normalize both score lists to [0, 1] per query  
2. Blend: `final = α × dense_norm + (1−α) × bm25_norm`  
3. Sweep α from 0.1 → 0.9 and pick the best NDCG@10

In [ ]:
def minmax_normalize(score_list):
    """Normalize a list of (doc_id, score) to [0, 1]."""
    if not score_list:
        return {}
    scores = [s for _, s in score_list]
    lo, hi = min(scores), max(scores)
    if hi == lo:
        return {doc_id: 1.0 for doc_id, _ in score_list}
    return {doc_id: (s - lo) / (hi - lo) for doc_id, s in score_list}


def alpha_fusion(bm25_ranked, dense_ranked, alpha=0.7, top_k=10):
    """
    Normalize each retriever's scores to [0,1] per query,
    then blend: final = alpha * dense + (1-alpha) * bm25
    Returns: {qid: {doc_id: blended_score}}
    """
    all_qids = set(bm25_ranked) | set(dense_ranked)
    results  = {}
    for qid in all_qids:
        bm25_norm  = minmax_normalize(bm25_ranked.get(qid,  []))
        dense_norm = minmax_normalize(dense_ranked.get(qid, []))
        all_docs   = set(bm25_norm) | set(dense_norm)
        blended    = {
            doc: alpha * dense_norm.get(doc, 0.0) + (1 - alpha) * bm25_norm.get(doc, 0.0)
            for doc in all_docs
        }
        top_docs = sorted(blended.items(), key=lambda x: x[1], reverse=True)[:top_k]
        results[qid] = dict(top_docs)
    return results


def quick_eval(results_dict, qrels_dict):
    """Return NDCG@10, MRR, Recall@10."""
    ev  = pytrec_eval.RelevanceEvaluator(qrels_dict, {"ndcg_cut_10", "recip_rank", "recall_10"})
    res = ev.evaluate(results_dict)
    return {
        "NDCG@10"   : round(np.mean([v["ndcg_cut_10"] for v in res.values()]), 4),
        "MRR"       : round(np.mean([v["recip_rank"]  for v in res.values()]), 4),
        "Recall@10" : round(np.mean([v["recall_10"]   for v in res.values()]), 4),
    }


# ── Alpha sweep ──────────────────────────────────────────────────────────────
alphas        = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
alpha_results = {}

print("Sweeping alpha (dense weight) ...")
print(f"  {'Alpha':>6}  {'NDCG@10':>8}  {'MRR':>8}  {'Recall@10':>10}")
print("  " + "-"*38)
for alpha in alphas:
    fused  = alpha_fusion(bm25_ranked, dense_ranked, alpha=alpha, top_k=10)
    metrics = quick_eval(fused, qrels_dict)
    alpha_results[alpha] = {"metrics": metrics, "results": fused}
    print(f"  {alpha:>6.1f}  {metrics['NDCG@10']:>8.4f}  {metrics['MRR']:>8.4f}  {metrics['Recall@10']:>10.4f}")

# Pick best alpha by NDCG@10
best_alpha   = max(alpha_results, key=lambda a: alpha_results[a]["metrics"]["NDCG@10"])
best_metrics = alpha_results[best_alpha]["metrics"]
best_results = alpha_results[best_alpha]["results"]
print(f"\n  Best alpha = {best_alpha}  →  NDCG@10 = {best_metrics['NDCG@10']}")

In [ ]:
# Save best alpha results
best_rows = [
    {"query_id": qid, "rank": rank + 1, "doc_id": doc_id, "score": round(score, 8)}
    for qid, docs in best_results.items()
    for rank, (doc_id, score) in enumerate(
        sorted(docs.items(), key=lambda x: x[1], reverse=True)
    )
]
pd.DataFrame(best_rows).to_csv(
    os.path.join(HYBRID_DIR, f"hybrid_alpha{best_alpha}_results.csv"), index=False
)
print(f"Saved: hybrid_alpha{best_alpha}_results.csv")

# Update hybrid_metrics.json with all results
with open(BM25_METRICS)  as f: bm25_m  = json.load(f)
with open(DENSE_METRICS) as f: dense_m = json.load(f)

full_comparison = {
    "dataset"     : "FiQA Test",
    "num_queries" : 648,
    "models": {
        "BM25 Baseline"              : {"NDCG@10": bm25_m["NDCG@10"],  "MRR": bm25_m["MRR"],  "Recall@10": bm25_m["Recall@10"]},
        "MiniLM-Dense"               : dense_m["models"]["MiniLM-Dense"],
        "Hybrid-RRF (k=60)"          : hybrid_agg,
        f"Hybrid-Alpha (α={best_alpha})": best_metrics,
    },
    "alpha_sweep" : {str(a): alpha_results[a]["metrics"] for a in alphas}
}
with open(os.path.join(HYBRID_DIR, "hybrid_metrics.json"), "w") as f:
    json.dump(full_comparison, f, indent=2)
print("Updated: hybrid_metrics.json")

# ── Final side-by-side comparison ────────────────────────────────────────────
bm25_ndcg = bm25_m["NDCG@10"]
print("\n" + "="*70)
print("            WEEK 2 — COMPLETE RESULTS (ALL METHODS)")
print("="*70)
print(f"  {'Model':<30} {'NDCG@10':>8} {'MRR':>8} {'Recall@10':>10}  {'vs BM25':>8}")
print("-"*70)
for name, m in full_comparison["models"].items():
    delta = f"+{((m['NDCG@10']-bm25_ndcg)/bm25_ndcg*100):.1f}%" if name != "BM25 Baseline" else "baseline"
    marker = " ◀ BEST" if m["NDCG@10"] == max(v["NDCG@10"] for v in full_comparison["models"].values()) else ""
    print(f"  {name:<30} {m['NDCG@10']:>8.4f} {m['MRR']:>8.4f} {m['Recall@10']:>10.4f}  {delta:>8}{marker}")
print("="*70)
print(f"\nBest hybrid method → α={best_alpha} — target to beat in Week 3 (cross-encoder re-ranking)")